# Assignment Beta - Question 3

**Name:** Muhammad Sabeeh  
**Roll number:** 23K-0002  
**Course:** Agentic AI (AI4015)  
**Semester:** Fall 2026


# Assignment Beta - Question 3
Student roll: 23K-0002. Natural logarithms throughout. This notebook contains executed calculations.

In [1]:
import math, statistics, json
from pathlib import Path
ROOT=Path.cwd()
ROLL="23K-0002"
def softmax(z, temperature=1):
    a=[math.exp((v-max(z))/temperature) for v in z]
    return [v/sum(a) for v in a]


def distill_loss(z):
    p=softmax([5,3,2,0],2); q=softmax(z,2); q1=softmax(z)
    terms=[a*math.log(a/b) for a,b in zip(p,q)]
    kl=sum(terms); ce=-math.log(q1[0])
    return {'teacher_t1':softmax([5,3,2,0]),'student_t1':q1,
            'teacher_t2':p,'student_t2':q,'kl_terms':terms,'kl':kl,'ce':ce,'loss':2*kl+.5*ce}


def q3():
    z=[3,2.5,1,.5]; x=distill_loss(z)
    grad=[q-p+.5*(q1-y) for p,q,q1,y in zip(x['teacher_t2'],x['student_t2'],x['student_t1'],[1,0,0,0])]
    new=[v-.5*g for v,g in zip(z,grad)]
    # Central difference checks the entire mixed objective, including T^2.
    numeric=[]
    for i in range(4):
        a=z.copy(); b=z.copy(); a[i]+=1e-5; b[i]-=1e-5
        numeric.append((distill_loss(a)['loss']-distill_loss(b)['loss'])/2e-5)
    assert max(abs(a-b) for a,b in zip(grad,numeric))<1e-8
    assert abs(sum(grad))<1e-12
    return {**x,'gradient':grad,'finite_difference_gradient':numeric,'updated_logits':new,'after_step':distill_loss(new)}


## (a)-(e) Distributions and loss
Softmax uses exp(z/T) normalized by its sum. L = 0.5 × 2² × KL(p₂ || q₂) + 0.5 × CE(y,q₁).

In [2]:
result=q3()
for key in ["teacher_t1","student_t1","teacher_t2","student_t2","kl_terms","kl","ce","loss"]:
    print(key, result[key])

teacher_t1 [0.8390245074625321, 0.11354961935990124, 0.04177257051535046, 0.00565330266221633]
student_t1 [0.5482603600826605, 0.33253671789522204, 0.07419897111919413, 0.045003950902923445]
teacher_t2 [0.5976948344902959, 0.21987964170334776, 0.13336374413970897, 0.04906177966664736]
student_t2 [0.4109839536767612, 0.3200746249532436, 0.15119254720903685, 0.11774887416095825]
kl_terms [0.22385234039287338, -0.08255905738886113, -0.016733663508391845, -0.04295230553020366]
kl 0.08160731396541676
ce 0.601004995223079
loss 0.46371712554237304


## (f) Gradient
dL/dzⱼ = αT(q₂ⱼ-p₂ⱼ) + (1-α)(q₁ⱼ-yⱼ). The central-difference check verifies the complete objective.

In [3]:
print("Analytic gradient:",result["gradient"])
print("Finite differences:",result["finite_difference_gradient"])
print("Gradient sum:",sum(result["gradient"]))

Analytic gradient: [-0.41258070077220443, 0.2664633421975069, 0.054928288628924946, 0.09118906994577261]
Finite differences: [-0.41258070076488623, 0.26646334219115886, 0.05492828863418175, 0.09118906994787233]
Gradient sum: 1.3877787807814457e-17


## (g) One update
z_new = z_old - 0.5 × gradient. Recompute both student distributions before recomputing the loss.

In [4]:
print("Updated logits:",result["updated_logits"])
print("Updated distributions and loss:",json.dumps(result["after_step"],indent=2))

Updated logits: [3.2062903503861024, 2.3667683289012467, 0.9725358556855376, 0.4544054650271137]
Updated distributions and loss: {
  "teacher_t1": [
    0.8390245074625321,
    0.11354961935990124,
    0.04177257051535046,
    0.00565330266221633
  ],
  "student_t1": [
    0.6238887483690329,
    0.2694681073088143,
    0.06683437673178466,
    0.03980876759036825
  ],
  "teacher_t2": [
    0.5976948344902959,
    0.21987964170334776,
    0.13336374413970897,
    0.04906177966664736
  ],
  "student_t2": [
    0.44700617421078864,
    0.2937741856885104,
    0.14630524519684385,
    0.11291439490385712
  ],
  "kl_terms": [
    0.17363507504037667,
    -0.06370596668665242,
    -0.012351462657334113,
    -0.0408954293141386
  ],
  "kl": 0.05668221638225154,
  "ce": 0.4717832143780207,
  "loss": 0.3492560399535134
}
